# Crop AP-Ratio Prediction  
**Pipeline:** Feature Engineering → CSV Export → Regression Modelling  
**Splits:** 70 / 15 / 15 (random) · Spatial OOD · Temporal OOD


In [1]:
# ─── Standard imports ───────────────────────────────────────────────────────
import warnings, os
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error)
from sklearn.linear_model import Ridge
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor)
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

pd.set_option("display.max_columns", None)
print("All imports OK")


All imports OK


## 1. Load Raw Data

In [12]:
DATA_PATH = "../dataset/preprocessed_data.csv"

df_raw = pd.read_csv(DATA_PATH)
print(f"Shape: {df_raw.shape}")
df_raw.head()


Shape: (4190, 15)


,Area,AP Ratio,District,Season,Avg Temp,Avg Humidity,Crop Name,Transplant,Growth,Harvest,Max Temp,Min Temp,Humidity Min,Humidity Max,Humidity Range
0,177321,0.851027,Bagerhat,Kharif 2,26.0,72.5,Aman,Jun,Jul to Oct,Nov to Dec,40,12.0,60,85,25
1,25646,1.175778,Bandarban,Kharif 2,26.0,72.5,Aman,Jun,Jul to Oct,Nov to Dec,40,12.0,60,85,25
2,231401,0.770589,Barguna,Kharif 2,26.0,72.5,Aman,Jun,Jul to Oct,Nov to Dec,40,12.0,60,85,25
3,302665,0.757104,Barishal,Kharif 2,26.0,72.5,Aman,Jun,Jul to Oct,Nov to Dec,40,12.0,60,85,25
4,388575,1.100652,Bhola,Kharif 2,26.0,72.5,Aman,Jun,Jul to Oct,Nov to Dec,40,12.0,60,85,25


## 2. Quick Data Audit

In [13]:
print("dtypes")
print(df_raw.dtypes)
print()
print("nulls")
print(df_raw.isnull().sum())
print()
print("numeric summary")
df_raw.describe()


dtypes
Area                int64
AP Ratio          float64
District              str
Season                str
Avg Temp          float64
Avg Humidity      float64
Crop Name             str
Transplant            str
Growth                str
Harvest               str
Max Temp            int64
Min Temp          float64
Humidity Min        int64
Humidity Max        int64
Humidity Range      int64
dtype: object

nulls
Area              0
AP Ratio          0
District          0
Season            0
Avg Temp          0
Avg Humidity      0
Crop Name         0
Transplant        0
Growth            0
Harvest           0
Max Temp          0
Min Temp          0
Humidity Min      0
Humidity Max      0
Humidity Range    0
dtype: int64

numeric summary


,Area,AP Ratio,Avg Temp,Avg Humidity,Max Temp,Min Temp,Humidity Min,Humidity Max,Humidity Range
count,4190.000000,4190.000000,4190.000000,4190.000000,4190.000000,4190.000000,4190.000000,4190.000000,4190.000000
mean,8891.410979,4.805791,23.967243,72.057279,30.866587,17.067900,62.166348,81.948210,19.781862
std,44126.335668,17.960122,4.690774,10.758557,5.894400,5.412313,12.125262,11.002829,8.554665
min,1.000000,0.000000,11.500000,35.000000,15.000000,5.000000,20.000000,45.000000,5.000000
25%,105.000000,1.279214,20.000000,67.500000,27.000000,14.000000,60.000000,80.000000,10.000000
50%,334.000000,2.523096,25.000000,72.500000,30.000000,18.000000,60.000000,85.000000,20.000000
75%,943.000000,4.717304,27.500000,80.000000,35.000000,20.000000,70.000000,90.000000,25.000000
max,663734.000000,951.216667,34.500000,90.000000,47.000000,27.000000,85.000000,100.000000,45.000000


In [14]:
print("Unique Districts :", df_raw["District"].nunique())
print("Unique Seasons :", df_raw["Season"].unique())
print("Unique Crops :", df_raw["Crop Name"].nunique())


Unique Districts : 64
Unique Seasons : <StringArray>
['Kharif 2', 'Kharif 1', 'Rabi']
Length: 3, dtype: str
Unique Crops : 72


## 3. Feature Engineering
We derive:
- **Thermal features** – diurnal range, growing-degree proxy, temp mean
- **Humidity features** – mid-range humidity, vapour pressure proxy
- **Crop-calendar ordinals** – month ordinals for transplant / growth-start / harvest
- **Season ordinal** – Rabi / Kharif 1 / Kharif 2 / Boro rank
- **Aggregated target statistics** – District-level & Crop-level mean / std AP Ratio (computed on training fold only to avoid leakage; here we build the columns on the full set and re-compute per fold where needed)
- **Interaction terms** – temp × humidity, area × thermal range


In [15]:
df = df_raw.copy()

# Month mapping
MONTH_MAP = {
    "Jan": 1,  "Feb": 2,  "Mar": 3,  "Apr": 4,
    "May": 5,  "Jun": 6,  "Jul": 7,  "Aug": 8,
    "Sep": 9,  "Oct": 10, "Nov": 11, "Dec": 12,
}

def month_ordinal(s: str) -> float:
    """Return the first month number found in a string like 'Jul to Oct'."""
    if pd.isna(s):
        return np.nan
    for tok in str(s).split():
        tok = tok.strip().rstrip(",")
        if tok in MONTH_MAP:
            return float(MONTH_MAP[tok])
    return np.nan

df["transplant_month"] = df["Transplant"].apply(month_ordinal)
df["growth_start_month"] = df["Growth"].apply(month_ordinal)
df["harvest_start_month"]= df["Harvest"].apply(month_ordinal)

# ── Thermal features ─────────────────────────────────────────────────────────
df["temp_diurnal_range"] = df["Max Temp"] - df["Min Temp"]
df["temp_mean"] = (df["Max Temp"] + df["Min Temp"]) / 2
# Growing degree days proxy (base 10 °C)
df["gdd_proxy"] = (df["temp_mean"] - 10).clip(lower=0)

# ── Humidity features ────────────────────────────────────────────────────────
df["humidity_mid"] = (df["Humidity Max"] + df["Humidity Min"]) / 2
df["humidity_range"] = df["Humidity Range"]          # already present
# Saturation vapour pressure proxy (Magnus approx)
df["svp_proxy"] = 0.611 * np.exp(17.27 * df["Avg Temp"] / (df["Avg Temp"] + 237.3))
df["vpd_proxy"] = df["svp_proxy"] * (1 - df["Avg Humidity"] / 100)

# ── Season ordinal ────────────────────────────────────────────────────────────
SEASON_ORDER = {"Rabi": 1, "Kharif 1": 2, "Kharif 2": 3, "Boro": 4}
df["season_ordinal"] = df["Season"].map(SEASON_ORDER).fillna(0).astype(int)

# ── Temporal features: extract year from Season string if present ─────────────
# e.g. "Kharif 2 2018"  →  year 2018  (skip if no year is embedded)
def extract_year(s):
    import re
    m = re.search(r"\b(19|20)\d{2}\b", str(s))
    return int(m.group()) if m else np.nan

# df["season_year"] = df["Season"].apply(extract_year)
# has_year = df["season_year"].notna().sum()
# print(f"Rows with year in Season string: {has_year} / {len(df)}")

# ── Interaction terms ─────────────────────────────────────────────────────────
df["area_x_diurnal"] = df["Area"] * df["temp_diurnal_range"]
df["temp_x_humidity"] = df["Avg Temp"] * df["Avg Humidity"]
df["area_log"] = np.log1p(df["Area"])

print("Feature engineering done. New shape:", df.shape)
df.head(3)


Feature engineering done. New shape: (4190, 29)


,Area,AP Ratio,District,Season,Avg Temp,Avg Humidity,Crop Name,Transplant,Growth,Harvest,Max Temp,Min Temp,Humidity Min,Humidity Max,Humidity Range,transplant_month,growth_start_month,harvest_start_month,temp_diurnal_range,temp_mean,gdd_proxy,humidity_mid,humidity_range,svp_proxy,vpd_proxy,season_ordinal,area_x_diurnal,temp_x_humidity,area_log
0,177321,0.851027,Bagerhat,Kharif 2,26.0,72.5,Aman,Jun,Jul to Oct,Nov to Dec,40,12.0,60,85,25,6.0,7.0,11.0,28.0,26.0,16.0,72.5,25,3.36254,0.924699,3,4964988.0,1885.0,12.085723
1,25646,1.175778,Bandarban,Kharif 2,26.0,72.5,Aman,Jun,Jul to Oct,Nov to Dec,40,12.0,60,85,25,6.0,7.0,11.0,28.0,26.0,16.0,72.5,25,3.36254,0.924699,3,718088.0,1885.0,10.152182
2,231401,0.770589,Barguna,Kharif 2,26.0,72.5,Aman,Jun,Jul to Oct,Nov to Dec,40,12.0,60,85,25,6.0,7.0,11.0,28.0,26.0,16.0,72.5,25,3.36254,0.924699,3,6479228.0,1885.0,12.351912


## 4. Encode Categorical Variables

In [16]:
CAT_COLS = ["Season", "Crop Name", "Transplant", "Growth", "Harvest"]

label_encoders = {}
for col in CAT_COLS:
    le = LabelEncoder()
    df[col + "_enc"] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

# District kept as-is for split logic; encode a copy for model input
le_district = LabelEncoder()
df["District_enc"] = le_district.fit_transform(df["District"].astype(str))

print("Encoded columns added.")


Encoded columns added.


## 5. Define Feature Matrix & Target

In [17]:
TARGET = "AP Ratio"

FEATURE_COLS = [
    # Raw numeric
    "Area", "area_log", "Avg Temp", "Avg Humidity",
    "Max Temp", "Min Temp", "Humidity Min", "Humidity Max", "Humidity Range",
    # Engineered
    "temp_diurnal_range", "temp_mean", "gdd_proxy",
    "humidity_mid", "svp_proxy", "vpd_proxy",
    "transplant_month", "growth_start_month", "harvest_start_month",
    "season_ordinal",
    "area_x_diurnal", "temp_x_humidity",
    # Encoded categoricals
    "Season_enc", "Crop Name_enc",
    "Transplant_enc", "Growth_enc", "Harvest_enc",
    "District_enc",
]

X = df[FEATURE_COLS].copy()
y = df[TARGET].copy()

print(f"Feature matrix : {X.shape}")
print(f"Target range : [{y.min():.4f}, {y.max():.4f}]  mean={y.mean():.4f}")


Feature matrix : (4190, 27)
Target range : [0.0000, 951.2167]  mean=4.8058


## 6. Export Engineered Dataset to CSV

In [19]:
OUT_CSV = "../dataset/crop_engineered.csv"
engineered_df = pd.concat([df[["District", "Season"]], X, y], axis=1)
engineered_df.to_csv(OUT_CSV, index=False)
print(f"Saved → {OUT_CSV}  ({engineered_df.shape[0]} rows × {engineered_df.shape[1]} cols)")


Saved → ../dataset/crop_engineered.csv  (4190 rows × 30 cols)


## 7. Train / Validation / Test Splits

Three evaluation regimes are defined:

| Split | Rationale |
|---|---|
| **Random 70/15/15** | Standard i.i.d. baseline |
| **Spatial OOD** | Disjoint district sets – tests geographic generalisation |
| **Temporal OOD** | Earlier seasons → train, later seasons → test (requires year info) |


In [20]:
SEED = 42

# A. Random split
X_tr_r, X_tmp_r, y_tr_r, y_tmp_r = train_test_split(X, y, test_size=0.30, random_state=SEED)
X_val_r, X_te_r, y_val_r, y_te_r = train_test_split(X_tmp_r, y_tmp_r, test_size=0.50, random_state=SEED)

print("Random Split")
print(f"  Train : {X_tr_r.shape[0]}  Val : {X_val_r.shape[0]}  Test : {X_te_r.shape[0]}")


Random Split
  Train : 2933  Val : 628  Test : 629


In [21]:
# ── B. Spatial OOD split ──────────────────────────────────────────────────────
np.random.seed(SEED)
all_districts = df["District"].unique()
np.random.shuffle(all_districts)

n_train_d = int(0.70 * len(all_districts))
n_val_d = int(0.15 * len(all_districts))

train_districts = all_districts[:n_train_d]
val_districts = all_districts[n_train_d:n_train_d + n_val_d]
test_districts = all_districts[n_train_d + n_val_d:]

mask_tr_s = df["District"].isin(train_districts)
mask_val_s = df["District"].isin(val_districts)
mask_te_s = df["District"].isin(test_districts)

X_tr_s, y_tr_s = X[mask_tr_s], y[mask_tr_s]
X_val_s, y_val_s = X[mask_val_s], y[mask_val_s]
X_te_s, y_te_s = X[mask_te_s], y[mask_te_s]

print("Spatial OOD Split")
print(f"  Train districts: {len(train_districts)}  Val: {len(val_districts)}  Test: {len(test_districts)}")
print(f"  Train rows: {X_tr_s.shape[0]}  Val: {X_val_s.shape[0]}  Test: {X_te_s.shape[0]}")


Spatial OOD Split
  Train districts: 44  Val: 9  Test: 11
  Train rows: 2879  Val: 587  Test: 724


In [ ]:
# ── C. Temporal OOD split (only when year info exists) ───────────────────────
if has_year > 0:
    df_t = df[df["season_year"].notna()].copy()
    X_t = X.loc[df_t.index]
    y_t = y.loc[df_t.index]
    years_sorted = sorted(df_t["season_year"].unique())
    n_y = len(years_sorted)
    cutoff_train = years_sorted[int(n_y * 0.70)]
    cutoff_val = years_sorted[int(n_y * 0.85)]

    mask_tr_t = df_t["season_year"] <= cutoff_train
    mask_val_t = (df_t["season_year"] > cutoff_train) & (df_t["season_year"] <= cutoff_val)
    mask_te_t = df_t["season_year"] > cutoff_val

    X_tr_t, y_tr_t = X_t[mask_tr_t.values], y_t[mask_tr_t.values]
    X_val_t, y_val_t = X_t[mask_val_t.values], y_t[mask_val_t.values]
    X_te_t, y_te_t = X_t[mask_te_t.values], y_t[mask_te_t.values]

    print("=== Temporal OOD Split ===")
    print(f"  Train years ≤ {cutoff_train}  |  Val ≤ {cutoff_val}  |  Test > {cutoff_val}")
    print(f"  Train: {len(X_tr_t)}  Val: {len(X_val_t)}  Test: {len(X_te_t)}")
else:
    print("No year info found in Season column — Temporal OOD split skipped.")
    X_tr_t = X_val_t = X_te_t = None


## 8. Feature Scaling

In [ ]:
def scale_split(X_tr, X_val, X_te):
    scaler = StandardScaler()
    Xs_tr = pd.DataFrame(scaler.fit_transform(X_tr), columns=X_tr.columns, index=X_tr.index)
    Xs_val = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns, index=X_val.index)
    Xs_te = pd.DataFrame(scaler.transform(X_te), columns=X_te.columns, index=X_te.index)
    return Xs_tr, Xs_val, Xs_te, scaler

Xs_tr_r, Xs_val_r, Xs_te_r, scaler_r = scale_split(X_tr_r, X_val_r, X_te_r)
Xs_tr_s, Xs_val_s, Xs_te_s, scaler_s = scale_split(X_tr_s, X_val_s, X_te_s)
if X_tr_t is not None:
    Xs_tr_t, Xs_val_t, Xs_te_t, scaler_t = scale_split(X_tr_t, X_val_t, X_te_t)

print("Scaling done.")


## 9. Model Training & Evaluation

In [ ]:
def metrics(y_true, y_pred, label=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    print(f"  {label:<12}  MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}  MAPE={mape:.2f}%")
    return dict(label=label, MAE=mae, RMSE=rmse, R2=r2, MAPE=mape)

def train_eval(name, model, Xs_tr, y_tr, Xs_val, y_val, Xs_te, y_te):
    model.fit(Xs_tr, y_tr)
    print(f"\n── {name} ──")
    val_m  = metrics(y_val, model.predict(Xs_val), "Val")
    test_m = metrics(y_te,  model.predict(Xs_te),  "Test")
    return model, val_m, test_m


In [ ]:
MODELS = {
    "Ridge" : Ridge(alpha=1.0),
    "RF" : RandomForestRegressor(n_estimators=200, max_depth=12, random_state=SEED, n_jobs=-1),
    "ExtraTrees": ExtraTreesRegressor(n_estimators=200, max_depth=12, random_state=SEED, n_jobs=-1),
    "GBM" : GradientBoostingRegressor(n_estimators=200, learning_rate=0.05,
                                            max_depth=5, random_state=SEED),
    "XGBoost" : XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                               subsample=0.8, colsample_bytree=0.8,
                               random_state=SEED, verbosity=0),
    "LightGBM" : LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=63,
                                random_state=SEED, verbosity=-1),
}


### 9.1 Random 70/15/15 Split Results

In [ ]:
print("\n══════════ RANDOM SPLIT ══════════")
random_results = {}
for name, model in MODELS.items():
    import copy
    m, val_m, test_m = train_eval(name, copy.deepcopy(model),
                                  Xs_tr_r, y_tr_r,
                                  Xs_val_r, y_val_r,
                                  Xs_te_r,  y_te_r)
    random_results[name] = {"model": m, "val": val_m, "test": test_m}


### 9.2 Spatial OOD Split Results

In [ ]:
print("\n══════════ SPATIAL OOD SPLIT ══════════")
spatial_results = {}
for name, model in MODELS.items():
    import copy
    m, val_m, test_m = train_eval(name, copy.deepcopy(model),
                                  Xs_tr_s, y_tr_s,
                                  Xs_val_s, y_val_s,
                                  Xs_te_s,  y_te_s)
    spatial_results[name] = {"model": m, "val": val_m, "test": test_m}


### 9.3 Temporal OOD Split Results (if year available)

In [ ]:
if X_tr_t is not None:
    print("\n══════════ TEMPORAL OOD SPLIT ══════════")
    temporal_results = {}
    for name, model in MODELS.items():
        import copy
        m, val_m, test_m = train_eval(name, copy.deepcopy(model),
                                      Xs_tr_t, y_tr_t,
                                      Xs_val_t, y_val_t,
                                      Xs_te_t,  y_te_t)
        temporal_results[name] = {"model": m, "val": val_m, "test": test_m}
else:
    print("Temporal OOD skipped — no year column.")


## 10. Consolidated Results Table

In [ ]:
def results_df(results_dict, split_name):
    rows = []
    for name, d in results_dict.items():
        rows.append({"Split": split_name, "Model": name, "Stage": "Val",  **{k: v for k,v in d["val"].items()  if k != "label"}})
        rows.append({"Split": split_name, "Model": name, "Stage": "Test", **{k: v for k,v in d["test"].items() if k != "label"}})
    return pd.DataFrame(rows)

rdf_random = results_df(random_results,  "Random")
rdf_spatial = results_df(spatial_results, "Spatial OOD")
all_results = pd.concat([rdf_random, rdf_spatial], ignore_index=True)

if X_tr_t is not None:
    rdf_temporal = results_df(temporal_results, "Temporal OOD")
    all_results = pd.concat([all_results, rdf_temporal], ignore_index=True)

all_results = all_results.sort_values(["Split","Stage","RMSE"]).reset_index(drop=True)
for col in ["MAE","RMSE","R2","MAPE"]:
    all_results[col] = all_results[col].round(4)

print(all_results.to_string(index=False))


## 11. Visualisations

In [ ]:
# ── 11a  R² bar chart ─────────────────────────────────────────────────────────
test_only = all_results[all_results["Stage"] == "Test"].copy()

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
splits = test_only["Split"].unique()
colors = sns.color_palette("Set2", len(MODELS))

for ax, split in zip(axes, splits):
    sub = test_only[test_only["Split"] == split].sort_values("R2", ascending=False)
    sns.barplot(data=sub, x="Model", y="R2", palette=colors, ax=ax)
    ax.set_title(f"{split} — Test R²", fontsize=12, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=30)
    for p in ax.patches:
        ax.annotate(f"{p.get_height():.3f}", (p.get_x() + p.get_width()/2, p.get_height()),
                    ha="center", va="bottom", fontsize=8)

plt.suptitle("Test R² Across Split Strategies", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("r2_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── 11b  Predicted vs Actual for best Random model ────────────────────────────
best_name = (test_only[test_only["Split"]=="Random"]
             .sort_values("R2", ascending=False)
             .iloc[0]["Model"])

best_model = random_results[best_name]["model"]
y_pred_best = best_model.predict(Xs_te_r)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_te_r, y_pred_best, alpha=0.4, s=20, color="steelblue")
lims = [min(y_te_r.min(), y_pred_best.min()), max(y_te_r.max(), y_pred_best.max())]
ax.plot(lims, lims, "r--", linewidth=1.5, label="Perfect prediction")
ax.set_xlabel("Actual AP Ratio")
ax.set_ylabel("Predicted AP Ratio")
ax.set_title(f"Predicted vs Actual — {best_name} (Random Test Set)")
ax.legend()
plt.tight_layout()
plt.savefig("pred_vs_actual.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── 11c  Feature importance for best tree-based model ────────────────────────
if hasattr(best_model, "feature_importances_"):
    imp = pd.Series(best_model.feature_importances_, index=FEATURE_COLS)
    top20 = imp.nlargest(20)

    fig, ax = plt.subplots(figsize=(8, 6))
    top20.sort_values().plot(kind="barh", ax=ax, color="teal")
    ax.set_title(f"Top-20 Feature Importances — {best_name}")
    ax.set_xlabel("Importance")
    plt.tight_layout()
    plt.savefig("feature_importance.png", dpi=150, bbox_inches="tight")
    plt.show()


In [ ]:
# ── 11d  RMSE heatmap across splits ──────────────────────────────────────────
pivot = (test_only.pivot_table(index="Model", columns="Split", values="RMSE"))
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(pivot, annot=True, fmt=".4f", cmap="YlOrRd", ax=ax)
ax.set_title("Test RMSE — Model × Split Strategy")
plt.tight_layout()
plt.savefig("rmse_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()


## 12. Residual Analysis

In [ ]:
residuals = y_te_r.values - y_pred_best

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.scatter(y_pred_best, residuals, alpha=0.4, s=15, color="coral")
ax1.axhline(0, color="black", linewidth=1)
ax1.set_xlabel("Predicted")
ax1.set_ylabel("Residual")
ax1.set_title("Residuals vs Predicted")

ax2.hist(residuals, bins=50, color="slateblue", edgecolor="white")
ax2.set_xlabel("Residual")
ax2.set_title("Residual Distribution")

plt.suptitle(f"Residual Analysis — {best_name} (Random Test)", fontweight="bold")
plt.tight_layout()
plt.savefig("residuals.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Residual mean: {residuals.mean():.6f}  std: {residuals.std():.4f}")


## 13. Generalisation Gap Analysis

In [ ]:
gap_rows = []
for name in MODELS.keys():
    for split, res in [("Random", random_results), ("Spatial OOD", spatial_results)]:
        if name in res:
            val_r2  = res[name]["val"]["R2"]
            test_r2 = res[name]["test"]["R2"]
            gap_rows.append({"Model": name, "Split": split,
                              "Val R²": val_r2, "Test R²": test_r2,
                              "Gap (Val−Test)": round(val_r2 - test_r2, 4)})

gap_df = pd.DataFrame(gap_rows)
print(gap_df.to_string(index=False))


## 14. Summary & Recommendations

In [ ]:
print("\n╔══════════════════════════════════════╗")
print("║        FINAL MODEL RECOMMENDATIONS   ║")
print("╚══════════════════════════════════════╝")

for split in test_only["Split"].unique():
    sub = test_only[test_only["Split"]==split].sort_values("R2", ascending=False)
    best = sub.iloc[0]
    print(f"\n  [{split}]")
    print(f"    Best model : {best['Model']}")
    print(f"    Test R²    : {best['R2']:.4f}")
    print(f"    Test RMSE  : {best['RMSE']:.4f}")
    print(f"    Test MAE   : {best['MAE']:.4f}")
    print(f"    Test MAPE  : {best['MAPE']:.2f}%")
